In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
#Load CV dataframe
keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping_update_calo.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_data.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_data_update_calo.df"
data_df = load_df(bnb_path, keys2load, 100)
data_pfp_df = data_df['pfp']
data_hit0_df = data_df['hit0']
data_hit1_df = data_df['hit1']
data_hit2_df = data_df['hit2']
data_hdr_df = data_df['hdr']

In [ ]:
mc_bnb_hit0_df.columns

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

if "pion" in bnb_path:
    data_tot_pot = 8.371e+19
else:
    data_tot_pot = 5.948e+18
    
print("data_tot_pot: %.3e" %(data_tot_pot))
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))

mc_bnb_pfp_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_pfp_df))
data_pfp_df[pot_weight_col] = np.ones(len(data_pfp_df))


In [ ]:
import pandas as pd


def analyze_track_tuples(hit_dfs_dict, pfp_df, dataset_name="Dataset"):
    """Extracts unique 4-tuple track/slice identifiers from hit and PFP DataFrames

    and computes union and overlap statistics.
    """
    target_levels = [
        "__ntuple",
        "entry",
        "rec.slc..index",
        "rec.slc.reco.pfp..index",
    ]

    print(f"=== Running analysis for {dataset_name} ===")

    # 1. Combined Unique Combinations Across Hit DataFrames
    hit_tuple_sets = []
    total_hit_rows = 0

    for plane, df in hit_dfs_dict.items():
        if df is not None and not df.empty:
            total_hit_rows += len(df)
            # Extract target index levels directly using Index.droplevel or Index.get_level_values
            # converting to MultiIndex / Index tuples directly is significantly faster than to_frame()
            tuples_set = set(df.index.droplevel(
                [col for col in df.index.names if col not in target_levels]
            ).unique())
            hit_tuple_sets.append(tuples_set)

    combined_hit_tuples = set.union(*hit_tuple_sets) if hit_tuple_sets else set()
    n_unique_combined_hits = len(combined_hit_tuples)

    print(f"Total hit rows (plane 0+1+2): {total_hit_rows}")
    print(f"Unique track identifiers in hits: {n_unique_combined_hits}")

    # 2. Unique Combinations for PFP DataFrame
    if pfp_df is not None and not pfp_df.empty:
        # Get unique index tuples for target levels
        pfp_tuples = set(pfp_df.index.droplevel(
            [col for col in pfp_df.index.names if col not in target_levels]
        ).unique())
        n_unique_pfp = len(pfp_tuples)
        total_pfp_rows = len(pfp_df)
    else:
        pfp_tuples = set()
        n_unique_pfp = 0
        total_pfp_rows = 0

    print(f"Total PFP rows: {total_pfp_rows}")
    print(f"Unique track identifiers in PFP: {n_unique_pfp}")

    # 3. Overlap between Hits and PFP
    if combined_hit_tuples and pfp_tuples:
        overlap = len(combined_hit_tuples.intersection(pfp_tuples))
        print(f"Unique tracks present in BOTH PFP and Hits: {overlap}")

    print("\n" + "=" * 50 + "\n")

    return {
        "hit_tuples": combined_hit_tuples,
        "pfp_tuples": pfp_tuples,
    }


# --- Define Input Data Structures ---

mc_bnb_hit_dfs = {
    "0": mc_bnb_hit0_df,
    "1": mc_bnb_hit1_df,
    "2": mc_bnb_hit2_df,
}

data_hit_dfs = {
    "0": data_hit0_df,
    "1": data_hit1_df,
    "2": data_hit2_df,
}

# --- Execute Analysis ---

mc_results = analyze_track_tuples(
    hit_dfs_dict=mc_bnb_hit_dfs,
    pfp_df=mc_bnb_pfp_df,
    dataset_name="MC BNB",
)

data_results = analyze_track_tuples(
    hit_dfs_dict=data_hit_dfs,
    pfp_df=data_pfp_df,
    dataset_name="Data",
)

In [ ]:
import pandas as pd


def calculate_ptype_breakdown(
    df,
    p_type_col,
    weight_col=None,
    dataset_name="Dataset",
    p_type_labels=None,
):
    """Calculates and prints unweighted and weighted particle type (p_type)

    yields and percentages for a given PFP DataFrame.
    """
    print("\n" + "=" * 65)
    print(f" Particle Composition Breakdown: {dataset_name} ")
    print("=" * 65)

    if df is None or df.empty:
        print(f"Warning: {dataset_name} DataFrame is empty or None.")
        return None

    # Check if p_type column exists
    if p_type_col not in df.columns:
        print(
            f"Error: Column {p_type_col} not found in {dataset_name} DataFrame."
        )
        return None

    # 1. Filter out missing p_type values
    valid_mask = df[p_type_col].notna()
    df_clean = df[valid_mask].copy()

    if df_clean.empty:
        print(f"Warning: No valid non-null entries found for {p_type_col}.")
        return None

    # 2. Extract weights if available, else default to 1.0
    if weight_col and weight_col in df_clean.columns:
        weights = df_clean[weight_col].fillna(1.0)
    else:
        weights = pd.Series(1.0, index=df_clean.index)

    # 3. Calculate Weighted & Unweighted Statistics
    stats_df = pd.DataFrame(
        {"p_type": df_clean[p_type_col], "weight": weights}
    )

    summary = (
        stats_df.groupby("p_type")
        .agg(Counts=("weight", "count"), Weighted_Yield=("weight", "sum"))
        .reset_index()
    )

    total_counts = summary["Counts"].sum()
    total_weighted = summary["Weighted_Yield"].sum()

    summary["Raw_%"] = (summary["Counts"] / total_counts) * 100
    summary["Weighted_%"] = (
        (summary["Weighted_Yield"] / total_weighted) * 100
        if total_weighted > 0
        else 0.0
    )

    # Map readable category labels if provided (e.g. {13: 'muon', 211: 'pion'})
    if p_type_labels:
        summary["p_type_label"] = summary["p_type"].map(
            lambda x: p_type_labels.get(x, str(x))
        )
    else:
        summary["p_type_label"] = summary["p_type"].astype(str)

    # Sort by weighted yield descending
    summary = summary.sort_values(by="Weighted_Yield", ascending=False)

    # 4. Pretty Print Output
    header = f"{'p_type':<15} | {'Counts':<8} | {'Raw %':<8} | {'Weighted':<10} | {'Weighted %':<10}"
    print(header)
    print("-" * len(header))

    for _, row in summary.iterrows():
        label = row["p_type_label"]
        print(
            f"{label:<15} | {int(row['Counts']):<8d} | {row['Raw_%']:<7.2f}% | "
            f"{row['Weighted_Yield']:<10.1f} | {row['Weighted_%']:<9.2f}%"
        )

    print("-" * len(header))
    print(
        f"{'Total':<15} | {total_counts:<8d} | {100.0:<7.2f}% | "
        f"{total_weighted:<10.1f} | {100.0:<9.2f}%"
    )
    print("=" * 65 + "\n")

    return summary


# --- Configuration ---

p_type_col = ("pfp", "trk", "truth", "p", "p_type", "")
weight_col = ("slc", "wgt", "", "", "", "")

# Optional label mapping dictionary for clean printing
p_type_labels_map = {
    0: "Other",
    1: "shower",
    13: "μ",
    2212: "p",
    211: "inelastic pion",
    -211: "stopping pion",
}


# --- Execute on Both DataFrames ---

mc_bnb_summary = calculate_ptype_breakdown(
    df=mc_bnb_pfp_df,
    p_type_col=p_type_col,
    weight_col=weight_col,
    dataset_name="MC BNB",
    p_type_labels=p_type_labels_map,  # Pass dictionary or None
)

data_summary = calculate_ptype_breakdown(
    df=data_pfp_df,
    p_type_col=p_type_col,
    weight_col=None,  # Data typically has no event weights
    dataset_name="Data",
    p_type_labels=p_type_labels_map,
)

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binned_statistic


def _compute_root_tprofile_stats(x_vals, y_vals, weights, bins_x):
    """Calculates ROOT-style TProfile statistics per bin:

    Weighted Mean <Y> and Standard Deviation sigma_Y.
    """
    bin_centers = 0.5 * (bins_x[:-1] + bins_x[1:])

    if len(x_vals) == 0:
        nan_arr = np.full_like(bin_centers, np.nan)
        return bin_centers, nan_arr, nan_arr, nan_arr

    # Binned sums
    sum_w_y, _, _ = binned_statistic(
        x_vals, y_vals * weights, statistic="sum", bins=bins_x
    )
    sum_w, _, _ = binned_statistic(
        x_vals, weights, statistic="sum", bins=bins_x
    )
    count, _, _ = binned_statistic(
        x_vals, weights, statistic="count", bins=bins_x
    )

    mean_y = np.full_like(sum_w, np.nan)
    valid_bins = sum_w > 0
    mean_y[valid_bins] = sum_w_y[valid_bins] / sum_w[valid_bins]

    # Weighted standard deviation (ROOT TProfile spread option 'S')
    sum_w_y2, _, _ = binned_statistic(
        x_vals, (y_vals**2) * weights, statistic="sum", bins=bins_x
    )

    std_y = np.full_like(sum_w, np.nan)
    std_err = np.full_like(sum_w, np.nan)

    variance = (sum_w_y2 / np.maximum(sum_w, 1e-9)) - (mean_y**2)
    variance = np.maximum(variance, 0.0)  # Numerical safety
    std_y[valid_bins] = np.sqrt(variance[valid_bins])

    # Standard Error on Mean: sigma / sqrt(N)
    std_err[valid_bins] = std_y[valid_bins] / np.sqrt(
        np.maximum(count[valid_bins], 1.0)
    )

    return bin_centers, mean_y, std_y, std_err


def _extract_clean_arrays(
    df, x_col, y_col, x_split_col, weight_col, y_max_cutoff=10.0
):
    """Safely filters non-null, finite values, and applies y_col <= y_max_cutoff mask."""
    if df is None or df.empty:
        return np.array([]), np.array([]), np.array([]), np.array([])

    mask = df[x_col].notna() & df[y_col].notna() & df[x_split_col].notna()
    plot_df = df[mask]

    x_vals = plot_df[x_col].values
    y_vals = plot_df[y_col].values
    split_vals = plot_df[x_split_col].values

    if weight_col and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0).values
    else:
        weights = np.ones_like(x_vals)

    finite_and_cutoff_mask = (
        np.isfinite(x_vals)
        & np.isfinite(y_vals)
        & np.isfinite(split_vals)
        & np.isfinite(weights)
        & (y_vals <= y_max_cutoff)
    )

    return (
        x_vals[finite_and_cutoff_mask],
        y_vals[finite_and_cutoff_mask],
        split_vals[finite_and_cutoff_mask],
        weights[finite_and_cutoff_mask],
    )


def plot_split_tpc_2d_with_root_profile(
    mc_df: pd.DataFrame,
    data_df: pd.DataFrame = None,
    x_col: str = "rr",
    y_col: str = "dedx",
    x_split_col: str = "x",
    mc_weight_col: str = None,
    data_weight_col: str = None,
    bins_x: np.ndarray = np.linspace(0, 150, 51),
    bins_y: np.ndarray = np.linspace(0, 10, 51),
    dedx_max_cutoff: float = 10.0,
    xlabel: str = "Residual Range [cm]",
    ylabel: str = "dE/dx [MeV/cm]",
    title_prefix: str = "Hit Distribution",
    cmap_name: str = "viridis",
    figsize: tuple = (16, 9),
):
    """Plots 2D histogram with ROOT-style TProfile in top plot

    and Data / MC ratio in the bottom plot.
    """
    mc_color = "#d62728"
    data_color = "#0f4c81"  # Dark navy blue

    # Styling settings for points and lines
    marker_size = 5.0
    line_width = 1.8
    cap_size = 3
    cap_thick = 1.5

    # 1. Extract Clean Data Arrays
    x_mc, y_mc, split_mc, w_mc = _extract_clean_arrays(
        mc_df, x_col, y_col, x_split_col, mc_weight_col, dedx_max_cutoff
    )
    x_data, y_data, split_data, w_data = _extract_clean_arrays(
        data_df, x_col, y_col, x_split_col, data_weight_col, dedx_max_cutoff
    )

    mc_neg, mc_pos = split_mc < 0, split_mc >= 0
    data_neg, data_pos = split_data < 0, split_data >= 0

    # 2. Setup Figure Layout with horizontal spacing (wspace=0.22)
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(
        2,
        3,
        height_ratios=[3, 1],
        width_ratios=[1, 1, 0.05],
        hspace=0.08,
        wspace=0.22,
    )

    ax_top_l = fig.add_subplot(gs[0, 0])
    ax_top_r = fig.add_subplot(gs[0, 1], sharey=ax_top_l)

    ax_bot_l = fig.add_subplot(gs[1, 0], sharex=ax_top_l)
    ax_bot_r = fig.add_subplot(gs[1, 1], sharex=ax_top_r, sharey=ax_bot_l)

    # Ensure right panel y-tick labels remain visible with wider separation
    plt.setp(ax_top_r.get_yticklabels(), visible=True)
    plt.setp(ax_bot_r.get_yticklabels(), visible=True)

    cax = fig.add_subplot(gs[0, 2])  # Colorbar axis

    cmap = globals().get("sunset_cmap", cmap_name)

    # Background Setup
    bg_x = x_mc if len(x_mc) > 0 else x_data
    bg_y = y_mc if len(y_mc) > 0 else y_data
    bg_split = split_mc if len(split_mc) > 0 else split_data
    bg_w = w_mc if len(w_mc) > 0 else w_data

    bg_neg, bg_pos = bg_split < 0, bg_split >= 0

    h_l, _, _ = np.histogram2d(
        bg_x[bg_neg], bg_y[bg_neg], bins=[bins_x, bins_y], weights=bg_w[bg_neg]
    )
    h_r, _, _ = np.histogram2d(
        bg_x[bg_pos], bg_y[bg_pos], bins=[bins_x, bins_y], weights=bg_w[bg_pos]
    )
    vmax = max(h_l.max(), h_r.max()) if max(h_l.max(), h_r.max()) > 0 else None

    # Top Plot 2D Background
    im0 = ax_top_l.hist2d(
        bg_x[bg_neg],
        bg_y[bg_neg],
        bins=[bins_x, bins_y],
        weights=bg_w[bg_neg],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]
    im1 = ax_top_r.hist2d(
        bg_x[bg_pos],
        bg_y[bg_pos],
        bins=[bins_x, bins_y],
        weights=bg_w[bg_pos],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]

    sides = [
        (ax_top_l, ax_bot_l, mc_neg, data_neg, "$X < 0$ cm"),
        (ax_top_r, ax_bot_r, mc_pos, data_pos, "$X \\geq 0$ cm"),
    ]

    for ax_top, ax_bot, mask_mc, mask_data, label in sides:
        # Calculate Stats (ROOT-style TProfile)
        cx, my_mc, std_mc, err_mc = _compute_root_tprofile_stats(
            x_mc[mask_mc], y_mc[mask_mc], w_mc[mask_mc], bins_x
        )
        _, my_data, std_data, err_data = _compute_root_tprofile_stats(
            x_data[mask_data], y_data[mask_data], w_data[mask_data], bins_x
        )

        # --- Top Plot: ROOT-style TProfile ---
        if len(x_mc[mask_mc]) > 0:
            ax_top.errorbar(
                cx,
                my_mc,
                yerr=std_mc,
                fmt="o",
                color=mc_color,
                ms=marker_size,
                elinewidth=line_width,
                capsize=cap_size,
                capthick=cap_thick,
                label="MC",
                zorder=10,
            )

        if len(x_data[mask_data]) > 0:
            ax_top.errorbar(
                cx,
                my_data,
                yerr=std_data,
                fmt="o",
                color=data_color,
                ms=marker_size,
                elinewidth=line_width,
                capsize=cap_size,
                capthick=cap_thick,
                label="Data",
                zorder=11,
            )

        # Top Formatting
        ax_top.set_title(f"{title_prefix}: {label}", fontsize=13, pad=8)
        ax_top.set_xlim(bins_x[0], bins_x[-1])
        ax_top.set_ylim(bins_y[0], bins_y[-1])
        ax_top.grid(alpha=0.3, linestyle="--")
        plt.setp(ax_top.get_xticklabels(), visible=False)

        # Top Legend with simplified labels ("MC" and "Data")
        ax_top.legend(
            loc="upper right",
            frameon=True,
            facecolor="white",
            framealpha=0.85,
            fontsize=15,
        )

        # --- Bottom Plot: Data / MC Ratio ---
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.divide(my_data, my_mc)
            ratio_err = np.divide(std_data, my_mc)  # Data_err / MC
            mc_rel_err = np.divide(std_mc, my_mc)   # Relative MC spread

        # 1. Draw MC Shaded Band around 1.0 representing relative MC spread
        valid_bins = np.isfinite(ratio) & np.isfinite(mc_rel_err)
        ax_bot.fill_between(
            cx,
            1.0 - mc_rel_err,
            1.0 + mc_rel_err,
            where=valid_bins,
            color=mc_color,
            alpha=0.25,
            zorder=8,
        )

        # 2. Draw Data / MC ratio points with Data_err / MC error bars
        ax_bot.errorbar(
            cx,
            ratio,
            yerr=ratio_err,
            fmt="o",
            color=data_color,
            ms=marker_size,
            elinewidth=line_width,
            capsize=cap_size,
            capthick=cap_thick,
            zorder=10,
        )
        ax_bot.axhline(1.0, color=mc_color, linestyle="--", linewidth=2.0)

        ax_bot.set_xlabel(xlabel, fontsize=12)
        ax_bot.grid(alpha=0.3, linestyle="--")

    # Y Labels
    ax_top_l.set_ylabel(ylabel, fontsize=12)
    ax_bot_l.set_ylabel("Data / MC", fontsize=11)

    # Colorbar
    cbar = fig.colorbar(im1, cax=cax)
    cbar.set_label("Weighted Entries", fontsize=11)

    # Hide extra bottom-right axis container
    ax_unused = fig.add_subplot(gs[1, 2])
    ax_unused.set_visible(False)

    return fig, (ax_top_l, ax_top_r, ax_bot_l, ax_bot_r)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define bin ranges for Residual Range and dE/dx
rr_bins = np.linspace(0, 80, 41)      # Residual Range [cm]
dedx_bins = np.linspace(0, 10, 51)     # dE/dx [MeV/cm]

# MultiIndex weight tuple (or string/None depending on your DF structure)
mc_weight = ('slc', 'wgt', '', '', '', '') 

# Loop through planes 0, 1, and 2
for plane_name in ['0', '1', '2']:
    mc_hit_df = mc_bnb_hit_dfs.get(plane_name)
    data_hit_df = data_hit_dfs.get(plane_name)

    fig, axes = plot_split_tpc_2d_with_root_profile(
        mc_df=mc_hit_df,
        data_df=data_hit_df,
        x_col="rr",
        y_col="dedx",
        x_split_col="x",
        mc_weight_col=mc_weight,
        data_weight_col=None,  # Data typically unweighted
        bins_x=rr_bins,
        bins_y=dedx_bins,
        xlabel="Residual Range [cm]",
        ylabel="Hit dE/dx [MeV/cm]",
        title_prefix=f"Plane {plane_name} Hit dE/dx vs RR",
    )
    

plt.show()

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_dedx_by_rr_and_tpc(
    df: pd.DataFrame,
    data_df: pd.DataFrame = None,
    rr_range: tuple = (0.0, 1.0),
    dedx_col: str = "dedx",
    rr_col: str = "rr",
    x_col: str = "x",
    weight_col: str = None,
    data_weight_col: str = None,
    bins: np.ndarray = np.linspace(0.0, 10.0, 51),
    stacked: bool = False,
    density: bool = False,
    alpha: float = 0.3,
    linewidth: float = 1.8,
    marker_size: float = 5.0,
    title: str = None,
    ax_top: plt.Axes = None,
    ax_bot: plt.Axes = None,
):
    """Plots dE/dx distribution for MC (histograms) and Data (points) 

    with corresponding Data/MC ratios per X region (X > 0 and X <= 0).
    """

    # --- Helper: Apply identical filter and extract split arrays ---
    def _extract_and_split(data_frame, w_col):
        if data_frame is None or data_frame.empty:
            return None, None

        # Filter identically for both MC and Data
        mask = (
            data_frame[dedx_col].notna()
            & data_frame[rr_col].notna()
            & data_frame[x_col].notna()
            & (data_frame[rr_col] >= rr_range[0])
            & (data_frame[rr_col] < rr_range[1])
        )
        plot_df = data_frame[mask]
        if plot_df.empty:
            return None, None

        if w_col and w_col in plot_df.columns:
            w = plot_df[w_col].fillna(1.0)
        else:
            w = pd.Series(1.0, index=plot_df.index)

        # Categorization by X Position
        mask_pos = plot_df[x_col] > 0
        mask_neg = ~mask_pos

        grouped_data = [
            plot_df.loc[mask_pos, dedx_col],
            plot_df.loc[mask_neg, dedx_col],
        ]
        grouped_weights = [
            w[mask_pos],
            w[mask_neg],
        ]

        return grouped_data, grouped_weights

    # --- 1. Filter Data & MC Identically ---
    mc_data, mc_weights = _extract_and_split(df, weight_col)
    if mc_data is None:
        raise ValueError(
            f"No valid MC entries found for {rr_col} in range {rr_range}."
        )

    data_data, data_weights = _extract_and_split(data_df, data_weight_col)

    # --- 2. Identical Density Transformation (density=True) ---
    bin_width = bins[1] - bins[0]
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    def _apply_density_transform(grouped_w, is_stacked):
        if not density or grouped_w is None:
            return grouped_w

        if is_stacked:
            total_w = sum(w.sum() for w in grouped_w)
            if total_w > 0:
                scale = 1.0 / (total_w * bin_width)
                return [w * scale for w in grouped_w]
        else:
            normed = []
            for w in grouped_w:
                w_sum = w.sum()
                normed.append(
                    w / (w_sum * bin_width) if w_sum > 0 else w
                )
            return normed
        return grouped_w

    # Apply same density transform logic to both MC and Data
    mc_weights = _apply_density_transform(mc_weights, stacked)
    data_weights = _apply_density_transform(data_weights, False)

    colors = ["#1f77b4", "#d62728"]  # Blue ($X > 0$) & Red ($X \le 0$)
    labels = [r"MC: $X > 0$ cm", r"MC: $X \leq 0$ cm"]
    data_labels = [r"Data: $X > 0$ cm", r"Data: $X \leq 0$ cm"]

    # --- 3. Figure Layout Setup ---
    if ax_top is None or ax_bot is None:
        fig, (ax_top, ax_bot) = plt.subplots(
            2,
            1,
            figsize=(8, 6),
            sharex=True,
            gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08},
        )
    else:
        fig = ax_top.get_figure()

    # --- 4. Plot MC Histograms & Extract Individual MC Bins ---
    ax_top.hist(
        mc_data,
        bins=bins,
        weights=mc_weights,
        stacked=stacked,
        histtype="stepfilled",
        color=colors,
        alpha=alpha,
        label=labels,
    )

    ax_top.hist(
        mc_data,
        bins=bins,
        weights=mc_weights,
        stacked=stacked,
        histtype="step",
        color=colors,
        linewidth=linewidth,
    )

    # Compute unstacked individual MC bin counts for ratio division
    mc_unstacked_binned = []
    for idx in range(2):
        mc_c, _ = np.histogram(mc_data[idx], bins=bins, weights=mc_weights[idx])
        mc_unstacked_binned.append(mc_c)

    # --- 5. Plot Data Points & Calculate Specific Ratios ---
    if data_data is not None:
        for idx in range(2):  # idx 0: X > 0 (Blue), idx 1: X <= 0 (Red)
            d_vals = data_data[idx]
            d_w = data_weights[idx]

            # Compute Data binned values and errors
            data_c, _ = np.histogram(d_vals, bins=bins, weights=d_w)
            sum_w2, _ = np.histogram(d_vals, bins=bins, weights=d_w**2)
            data_err = np.sqrt(sum_w2)

            # Top plot: Data points
            valid_data = data_c > 0
            ax_top.errorbar(
                bin_centers[valid_data],
                data_c[valid_data],
                yerr=data_err[valid_data],
                fmt="o",
                color=colors[idx],
                ms=marker_size,
                elinewidth=linewidth,
                capsize=2,
                capthick=1.2,
                label=data_labels[idx],
                zorder=10 + idx,
            )

            # Bottom plot: Data / MC ratio per region
            # Data (X > 0) / MC (X > 0) AND Data (X <= 0) / MC (X <= 0)
            mc_c = mc_unstacked_binned[idx]

            with np.errstate(divide="ignore", invalid="ignore"):
                ratio = np.divide(data_c, mc_c)
                ratio_err = np.divide(data_err, mc_c)

            valid_ratio = np.isfinite(ratio) & (mc_c > 0)

            ax_bot.errorbar(
                bin_centers[valid_ratio],
                ratio[valid_ratio],
                yerr=ratio_err[valid_ratio],
                fmt="o",
                color=colors[idx],
                ms=marker_size,
                elinewidth=linewidth,
                capsize=2,
                capthick=1.2,
                zorder=10 + idx,
            )

    ax_bot.axhline(1.0, color="gray", linestyle="--", linewidth=1.5)

    # --- 6. Styling & Formatting ---
    ax_bot.set_xlabel(r"Hit $dE/dx$ [MeV/cm]", fontsize=13)

    if density:
        ax_top.set_ylabel("A.U.", fontsize=13)
    elif weight_col:
        ax_top.set_ylabel("Weighted Hits", fontsize=13)
    else:
        ax_top.set_ylabel("Hits", fontsize=13)

    ax_bot.set_ylabel("Data / MC", fontsize=11)

    if title is None:
        title = rf"Hit $dE/dx$ Distribution ({rr_range[0]} $\leq$ RR < {rr_range[1]} cm)"
    ax_top.set_title(title, fontsize=14, pad=10)

    ax_top.set_xlim(bins[0], bins[-1])
    ax_top.tick_params(axis="both", which="both", labelsize=11, direction="in")
    ax_bot.tick_params(axis="both", which="both", labelsize=11, direction="in")

    ax_top.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax_bot.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)

    ax_top.legend(
        fontsize=12,
        frameon=True,
        framealpha=1.0,
        edgecolor="black",
        fancybox=False,
    )

    return fig, (ax_top, ax_bot)

In [ ]:
# Inputs
mc_hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
data_hit_dfs = [
    data_hit0_df,
    data_hit1_df,
    data_hit2_df,
]  # Optional: Set to [None, None, None] if no Data available
rr_ranges = [(3, 4), (6, 7), (15, 16)]

n_planes = len(mc_hit_dfs)

for rr_min, rr_max in rr_ranges:
    fig = plt.figure(figsize=(18, 7))
    gs = gridspec.GridSpec(
        2,
        n_planes,
        height_ratios=[3, 1],
        hspace=0.08,
        wspace=0.15,
    )

    top_axes = []
    bot_axes = []

    # Create top and bottom axes per plane
    for plane_idx in range(n_planes):
        ax_top = fig.add_subplot(gs[0, plane_idx])
        ax_bot = fig.add_subplot(
            gs[1, plane_idx], sharex=ax_top, sharey=bot_axes[0] if plane_idx > 0 else None
        )

        top_axes.append(ax_top)
        bot_axes.append(ax_bot)

    max_y_value = 0.0  # Track global maximum height for top plots

    for plane_idx, (mc_df_plane, data_df_plane) in enumerate(
        zip(mc_hit_dfs, data_hit_dfs)
    ):
        ax_t = top_axes[plane_idx]
        ax_b = bot_axes[plane_idx]

        try:
            plot_dedx_by_rr_and_tpc(
                df=mc_df_plane,
                data_df=data_df_plane,
                rr_range=(rr_min, rr_max),
                dedx_col="dedx",
                rr_col="rr",
                x_col="x",
                weight_col=None,
                bins=np.linspace(0.0, 10.0, 26),
                stacked=False,
                density=True,
                title=f"Plane {plane_idx}",
                ax_top=ax_t,
                ax_bot=ax_b,
            )

            # Record max y for top plot alignment
            current_max = ax_t.get_ylim()[1] / 1.15
            if current_max > max_y_value:
                max_y_value = current_max

        except ValueError:
            ax_t.set_title(f"Plane {plane_idx}: No Data", fontsize=14, pad=10)
            ax_b.set_visible(False)
            continue

        # Hide redundant y-axis labels and legends on non-leftmost panels
        if plane_idx > 0:
            ax_t.set_ylabel("")
            ax_b.set_ylabel("")
            plt.setp(ax_t.get_yticklabels(), visible=True)
            plt.setp(ax_b.get_yticklabels(), visible=True)

            legend = ax_t.get_legend()
            if legend:
                legend.remove()

        # Hide top x-axis tick labels (sharex with ratio)
        plt.setp(ax_t.get_xticklabels(), visible=False)

    # Synchronize Y-limits across all top plots
    if max_y_value > 0:
        for ax_t in top_axes:
            ax_t.set_ylim(0, max_y_value * 1.15)

    # Synchronize Y-limits for bottom ratio plots (0.0 to 2.0 default)
    for ax_b in bot_axes:
        ax_b.set_ylim(0.0, 2.0)

    fig.suptitle(
        rf"Normalized Hit $dE/dx$ Comparison ({rr_min} $\leq$ RR < {rr_max} cm)",
        fontsize=16,
        y=0.98,
    )

    plt.show()

In [ ]:
hfit = load_physics_classes()

In [ ]:
import numpy as np
import pandas as pd
import ROOT


def make_rr_edges(rr_min, rr_max, bin_width):
    return np.arange(rr_min, rr_max + bin_width, bin_width)


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

PLANE_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c"]  # Blue, Orange, Green
PARAM_SPECS = [
    ("mpv_reco", "mpv_reco_err", "LanGau MPV", "MPV [MeV/cm]"),
    ("gsigma", "gsigma_err", r"Gaussian Smearing $\sigma_G$", r"$\sigma_G$ [MeV/cm]"),
    ("width", "width_err", r"Landau Scale Width $\xi$", r"$\xi$ [MeV/cm]"),
    ("area", "area_err", "Fit Area", "Area"),
]


def exp_decay(x, a, b, c):
    """Exponential decay function: f(x) = a * exp(-b * x) + c"""
    return a * np.exp(-b * x) + c


def power_law(x, a, b, c):
    """f(x) = a * x^(-b) + c"""
    return a * np.power(x, -b) + c


def fit_param_curve(res_df, param_col, err_col, rr_lo=5.0, rr_hi=None, model="power_law"):
    """Fits a decay curve (power law by default) to a parameter vs rr_center."""
    if len(res_df) < 3:
        return None, None

    mask = res_df["rr_center"] >= rr_lo
    if rr_hi is not None:
        mask &= res_df["rr_center"] <= rr_hi

    xs = res_df.loc[mask, "rr_center"].values
    ys = res_df.loc[mask, param_col].values
    yerrs = res_df.loc[mask, err_col].values

    valid = np.isfinite(xs) & np.isfinite(ys) & np.isfinite(yerrs) & (yerrs > 0) & (xs > 0)
    if np.sum(valid) < 3:
        return None, None

    x_val, y_val, y_err = xs[valid], ys[valid], yerrs[valid]

    if model == "power_law":
        func = power_law
        c_guess = np.percentile(y_val, 10)
        a_guess = max((np.percentile(y_val, 90) - c_guess) * (x_val.min() ** 1.0), 1e-3)
        p0 = [a_guess, 1.0, c_guess]
        bounds = ([0, 0, -np.inf], [np.inf, np.inf, np.inf])
    elif model == "double_exp":
        func = double_exp
        c_guess = np.percentile(y_val, 10)
        span = np.percentile(y_val, 90) - c_guess
        p0 = [span * 0.7, 0.3, span * 0.3, 0.02, c_guess]
        bounds = ([0, 0, 0, 0, -np.inf], [np.inf, np.inf, np.inf, np.inf, np.inf])
    elif model == "inverse":
        func = inverse_law
        c_guess = np.percentile(y_val, 10)
        a_guess = (np.percentile(y_val, 90) - c_guess) * x_val.min()
        p0 = [a_guess, c_guess]
        bounds = ([0, -np.inf], [np.inf, np.inf])
    else:
        raise ValueError(f"Unknown model: {model!r}")

    try:
        popt, pcov = curve_fit(
            func, x_val, y_val, sigma=y_err, absolute_sigma=True,
            p0=p0, bounds=bounds, maxfev=5000,
        )
        perr = np.sqrt(np.diag(pcov))
        return popt, perr
    except Exception:
        return None, None
'''
def analyze(
    hit_dfs,
    particle="muon",
    dedx_col="dedx",
    rr_max_by_particle=None,
    theoretical_mpv_func=None,
    out_prefix="langau_rr",
    make_summary_plots=True,
    verbose=True,
):
    """Analyzes hit DataFrames and performs exponential fits for MPV, gsigma, and width vs RR.

    Returns:
        all_results (dict): {(plane, tpc): DataFrame_per_slice}
        fit_params (dict):  {(plane, tpc): {param_name: (popt, perr)}}
        combined (pd.DataFrame): Master DataFrame of all slices, planes, and fits.
    """
    if rr_max_by_particle is None:
        rr_max_by_particle = {"muon": 80.0, "pion": 60.0, "proton": 60.0}
    rr_max = rr_max_by_particle.get(particle, 80.0)

    all_results = {}
    fit_params = {}

    for plane, df in enumerate(hit_dfs):
        for tpc in (0, 1, -1):
            if verbose:
                tpc_label = "combined" if tpc == -1 else tpc
                print(f"Plane {plane}, TPC {tpc_label}")

            sub = df if tpc == -1 else df[df["tpc"] == tpc]

            res = fit_rr_slices(
                sub,
                plane,
                tpc,
                dedx_col=dedx_col,
                rr_max=rr_max,
                theoretical_mpv_func=theoretical_mpv_func,
                verbose=verbose,
            )
            # Store full results (including < 3 cm slices)
            all_results[(plane, tpc)] = res

            # Filter out RR < 3 cm slices ONLY for the exponential fitting step
            res_fit = res[res["rr_center"] >= 3.0] if "rr_center" in res.columns else res

            # Fit exponential decay to MPV, GSigma, and Width vs Residual Range
            plane_fits = {}
            if len(res_fit) >= 3:
                for p_col, p_err, _, _ in PARAM_SPECS:
                    if p_col == "area":
                        continue  # Skip fit for Area as requested
                    popt, perr = fit_exp_param(res_fit, p_col, p_err)
                    plane_fits[p_col] = (popt, perr)

            fit_params[(plane, tpc)] = plane_fits

    non_empty = [
        r.assign(plane=p, tpc=t)
        for (p, t), r in all_results.items()
        if len(r)
    ]
    combined = (
        pd.concat(non_empty, ignore_index=True)
        if non_empty
        else pd.DataFrame()
    )

    if len(combined):
        combined.to_hdf(
            f"{out_prefix}_slices.h5",
            key="fits",
            mode="w",
            format="table",
            complib="blosc",
            complevel=9,
        )

    return all_results, fit_params, combined
'''
def analyze(
    hit_dfs,
    particle="muon",
    dedx_col="dedx",
    rr_max_by_particle=None,
    theoretical_mpv_func=None,
    out_prefix="langau_rr",
    make_summary_plots=True,
    verbose=True,
    fit_model="power_law",
    fit_rr_lo=5.0,
):
    """Analyzes hit DataFrames and performs power-law fits for MPV, gsigma, and width vs RR.

    Returns:
        all_results (dict): {(plane, tpc): DataFrame_per_slice}
        fit_params (dict):  {(plane, tpc): {param_name: (popt, perr)}}
        combined (pd.DataFrame): Master DataFrame of all slices, planes, and fits.
    """
    if rr_max_by_particle is None:
        rr_max_by_particle = {"muon": 80.0, "pion": 60.0, "proton": 60.0}
    rr_max = rr_max_by_particle.get(particle, 80.0)
    all_results = {}
    fit_params = {}
    for plane, df in enumerate(hit_dfs):
        for tpc in (0, 1, -1):
            if verbose:
                tpc_label = "combined" if tpc == -1 else tpc
                print(f"Plane {plane}, TPC {tpc_label}")
            sub = df if tpc == -1 else df[df["tpc"] == tpc]
            res = fit_rr_slices(
                sub,
                plane,
                tpc,
                dedx_col=dedx_col,
                rr_max=rr_max,
                theoretical_mpv_func=theoretical_mpv_func,
                verbose=verbose,
            )
            # Store full results (including the Bragg-peak-adjacent slices)
            all_results[(plane, tpc)] = res
            # Filter out slices too close to the Bragg peak ONLY for the curve-fitting step
            res_fit = res[res["rr_center"] >= fit_rr_lo] if "rr_center" in res.columns else res
            # Fit power law (default) to MPV, GSigma, and Width vs Residual Range
            plane_fits = {}
            if len(res_fit) >= 3:
                for p_col, p_err, _, _ in PARAM_SPECS:
                    if p_col == "area":
                        continue  # Skip fit for Area as requested
                    popt, perr = fit_param_curve(
                        res_fit, p_col, p_err, rr_lo=fit_rr_lo, model=fit_model
                    )
                    plane_fits[p_col] = (popt, perr)
            fit_params[(plane, tpc)] = plane_fits
    non_empty = [
        r.assign(plane=p, tpc=t)
        for (p, t), r in all_results.items()
        if len(r)
    ]
    combined = (
        pd.concat(non_empty, ignore_index=True)
        if non_empty
        else pd.DataFrame()
    )
    if len(combined):
        combined.to_hdf(
            f"{out_prefix}_slices.h5",
            key="fits",
            mode="w",
            format="table",
            complib="blosc",
            complevel=9,
        )
    return all_results, fit_params, combined
    
def fit_rr_slices(
    df,
    plane,
    tpc,
    dedx_col="dedx",
    rr_min=2.0,
    rr_max=40.0,
    rr_bin_width=1.0,
    hist_nbins=150,
    hist_xmin=0,
    hist_xmax=20.0,
    min_entries=30,
    first_stage_range=(0.5, 20.0),
    theoretical_mpv_func=None,
    max_gsigma_err=0.2,
    min_gsigma_err=0.0001,
    verbose=False,
):
    """Fits individual RR slices and collects MPV, Landau Width, Area, and Gaussian Smearing parameters."""
    edges = make_rr_edges(rr_min, rr_max, rr_bin_width)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sl = df[(df["rr"] >= lo) & (df["rr"] < hi)]
        if len(sl) < min_entries:
            continue

        rr_center = 0.5 * (lo + hi)
        hname = f"slice_p{plane}_t{tpc}_rr{rr_center:.2f}"
        hist = th1_from_series(
            sl[dedx_col], hname, "", hist_nbins, hist_xmin, hist_xmax
        )

        fit = langau_fit_two_stage(hist, first_stage_range=first_stage_range)
        if fit is None:
            if verbose:
                print(
                    f"   [plane {plane}, tpc {tpc}] rr={rr_center:.2f}: fit rejected"
                )
            continue

        width, mpv_reco, area, gsigma = fit["pars"]
        w_e, mpv_e, area_e, gsigma_e = fit["errs"]

        if gsigma_e > max_gsigma_err or not np.isfinite(gsigma_e) or gsigma_e < min_gsigma_err:
            if verbose:
                print(
                    f"   [plane {plane}, tpc {tpc}] rr={rr_center:.2f}: dropped due to gsigma_err={gsigma_e:.3f}"
                )
            continue

        if theoretical_mpv_func is not None:
            mean_pitch = sl["pitch"].mean()
            mpv_x = theoretical_mpv_func(rr_center, mean_pitch)
            mpv_x_err = 0.0
        else:
            mpv_x = mpv_reco
            mpv_x_err = mpv_e

        rows.append(
            dict(
                plane=plane,
                tpc=tpc,
                rr_center=rr_center,
                n_hits=len(sl),
                mpv_x=mpv_x,
                mpv_x_err=mpv_x_err,
                mpv_reco=mpv_reco,
                mpv_reco_err=mpv_e,
                width=width,
                width_err=w_e,
                area=area,
                area_err=area_e,
                gsigma=gsigma,
                gsigma_err=gsigma_e,
                chi2=fit["chi2"],
                ndf=fit["ndf"],
                status=fit["status"],
            )
        )

    return pd.DataFrame(rows)

In [ ]:

# ===========================================================================
# 7. Execution Driver Script
# ===========================================================================
pdg = 13
particle = "muon"
if "pion" in bnb_path:
    pdg = 211
    particle = "pion"

# 1. Theoretical MPV generator
theoretical_mpv = make_theoretical_mpv_func(hfit, pdg=pdg)

# 2. Analyze Data
data_results, data_params, data_df = analyze(
    data_hit_dfs,
    particle=particle,
    theoretical_mpv_func=theoretical_mpv,
    out_prefix="data_langau",
    make_summary_plots=False,
)

# 3. Analyze MC
mc_results, mc_params, mc_df = analyze(
    mc_hit_dfs,
    particle=particle,
    theoretical_mpv_func=theoretical_mpv,
    out_prefix="mc_langau",
    make_summary_plots=False,
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

PLANE_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c"]  # Blue, Orange, Green
PARAM_SPECS = [
    ("mpv_reco", "mpv_reco_err", "LanGau MPV", "MPV [MeV/cm]"),
    ("gsigma", "gsigma_err", r"Gaussian Smearing $\sigma_G$", r"$\sigma_G$ [MeV/cm]"),
    ("width", "width_err", r"Landau Scale Width $\xi$", r"$\xi$ [MeV/cm]"),
    ("area", "area_err", "Fit Area", "Area"),
]


def plot_all_planes_params_vs_rr(
    data_results,
    data_params,
    mc_results,
    mc_params,
    particle="muon",
    out_prefix="langau_rr",
    show_fits=True,
    fit_func=power_law,
):
    """Plots all 4 LanGau parameters vs Residual Range for all TPCs and Planes,

    overlaying the power-law fit curves where applicable.
    """
    tpc_list = [0, 1, -1]
    tpc_titles = {
        0: r"$X < 0$ (TPC 0)",
        1: r"$X > 0$ (TPC 1)",
        -1: "Both TPCs Combined",
    }

    n_rows = len(PARAM_SPECS)
    n_cols = len(tpc_list)

    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(16, 14), sharex=True, sharey="row"
    )

    for col_idx, tpc in enumerate(tpc_list):
        for row_idx, (p_col, p_err, p_title, p_ylabel) in enumerate(PARAM_SPECS):
            ax = axes[row_idx, col_idx]

            for plane in range(3):
                color = PLANE_COLORS[plane]

                # --- Data Points & Fit ---
                res_data = data_results.get((plane, tpc))
                if res_data is not None and len(res_data):
                    ax.errorbar(
                        res_data["rr_center"],
                        res_data[p_col],
                        yerr=res_data[p_err],
                        fmt="o",
                        color=color,
                        ms=4,
                        capsize=2,
                        alpha=0.7,
                        label=f"Data P{plane}",
                    )

                    if show_fits and p_col != "area":
                        popt_data, _ = data_params.get((plane, tpc), {}).get(p_col, (None, None))
                        if popt_data is not None:
                            xs = np.linspace(res_data["rr_center"].min(), res_data["rr_center"].max(), 200)
                            ax.plot(xs, fit_func(xs, *popt_data), "-", color=color, lw=1.5)

                # --- MC Points & Fit ---
                res_mc = mc_results.get((plane, tpc))
                if res_mc is not None and len(res_mc):
                    ax.errorbar(
                        res_mc["rr_center"],
                        res_mc[p_col],
                        yerr=res_mc[p_err],
                        fmt="s",
                        color=color,
                        ms=3,
                        markerfacecolor="none",
                        markeredgewidth=1.2,
                        capsize=2,
                        alpha=0.7,
                        label=f"MC P{plane}",
                    )

                    if show_fits and p_col != "area":
                        popt_mc, _ = mc_params.get((plane, tpc), {}).get(p_col, (None, None))
                        if popt_mc is not None:
                            xs = np.linspace(res_mc["rr_center"].min(), res_mc["rr_center"].max(), 200)
                            ax.plot(xs, fit_func(xs, *popt_mc), "--", color=color, lw=1.5)

            # Titles & Labels
            if row_idx == 0:
                ax.set_title(tpc_titles[tpc], fontsize=13, pad=10)
            if col_idx == 0:
                ax.set_ylabel(f"{p_ylabel}", fontsize=11)
            if row_idx == n_rows - 1:
                ax.set_xlabel("Residual Range [cm]", fontsize=11)

            ax.grid(True, linestyle=":", alpha=0.5)
            if row_idx == 0 and col_idx == 0:
                ax.legend(fontsize=7, loc="upper right", ncol=2, framealpha=0.9)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_all_params_vs_rr.pdf")
    plt.show()


def plot_by_plane_params_vs_rr(
    data_results,
    data_params,
    mc_results,
    mc_params,
    particle="muon",
    out_prefix="comparison",
    target_tpc=-1,
    show_fits=True,
    fit_func=power_law,
):
    """Creates individual 2x2 parameter summary plots for each plane (0, 1, 2)

    focusing ONLY on the specified TPC configuration (default: Combined TPC = -1).
    Overlays Data vs MC points alongside power-law fit curves.
    """
    color_data = "#1f77b4"  # Blue
    color_mc = "#d62728"    # Red

    tpc_label = "Both TPCs Combined" if target_tpc == -1 else f"TPC {target_tpc}"

    for plane in range(3):
        fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True)
        axes_flat = axes.flatten()

        for idx, (p_col, p_err, p_title, p_ylabel) in enumerate(PARAM_SPECS):
            ax = axes_flat[idx]

            # --- Data Points & Fit ---
            res_data = data_results.get((plane, target_tpc))
            if res_data is not None and len(res_data):
                ax.errorbar(
                    res_data["rr_center"],
                    res_data[p_col],
                    yerr=res_data[p_err],
                    fmt="o",
                    color=color_data,
                    ms=4,
                    capsize=2,
                    label="Data Points",
                )

                if show_fits and p_col != "area":
                    popt_data, _ = data_params.get((plane, target_tpc), {}).get(p_col, (None, None))
                    if popt_data is not None:
                        xs = np.linspace(res_data["rr_center"].min(), res_data["rr_center"].max(), 200)
                        ax.plot(
                            xs,
                            fit_func(xs, *popt_data),
                            "-",
                            color=color_data,
                            lw=1.8,
                            label=rf"Data Fit: ${popt_data[0]:.2f}x^{{-{popt_data[1]:.2f}}} + {popt_data[2]:.2f}$",
                        )

            # --- MC Points & Fit ---
            res_mc = mc_results.get((plane, target_tpc))
            if res_mc is not None and len(res_mc):
                ax.errorbar(
                    res_mc["rr_center"],
                    res_mc[p_col],
                    yerr=res_mc[p_err],
                    fmt="s",
                    color=color_mc,
                    ms=4,
                    markerfacecolor="none",
                    markeredgewidth=1.2,
                    capsize=2,
                    label="MC Points",
                )

                if show_fits and p_col != "area":
                    popt_mc, _ = mc_params.get((plane, target_tpc), {}).get(p_col, (None, None))
                    if popt_mc is not None:
                        xs = np.linspace(res_mc["rr_center"].min(), res_mc["rr_center"].max(), 200)
                        ax.plot(
                            xs,
                            fit_func(xs, *popt_mc),
                            "--",
                            color=color_mc,
                            lw=1.8,
                            label=rf"MC Fit: ${popt_mc[0]:.2f}x^{{-{popt_mc[1]:.2f}}} + {popt_mc[2]:.2f}$",
                        )

            ax.set_title(p_title, fontsize=12)
            ax.set_ylabel(p_ylabel, fontsize=11)
            ax.grid(True, linestyle=":", alpha=0.5)

            if idx >= 2:
                ax.set_xlabel("Residual Range [cm]", fontsize=11)
            ax.legend(fontsize=7, loc="best", framealpha=0.9)

        fig.suptitle(f"Plane {plane} LanGau Parameters vs Residual Range ({tpc_label})", fontsize=14, y=0.98)
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        fig.savefig(f"{out_prefix}_plane{plane}_combined_tpc.pdf")
        plt.show()

In [ ]:
# 4. Generate Data vs. MC comparison plots
# 4. Generate Data vs. MC comparison plots
plot_all_planes_params_vs_rr(
    data_results,
    data_params,
    mc_results,
    mc_params,
    particle=particle,
)

plot_by_plane_params_vs_rr(
    data_results,
    data_params,
    mc_results,
    mc_params,
    particle=particle,
    out_prefix="comparison",
)

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np


def power_law(x, a, b, c):
    """f(x) = a * x^(-b) + c"""
    return a * np.power(x, -b) + c


def _get_params_dict(params, plane_idx, tpc):
    """Robust key lookup in fit parameters dictionary."""
    if params is None:
        return None
    for k in [(plane_idx, tpc), plane_idx, (plane_idx, -1), (plane_idx, 0)]:
        if k in params:
            return params[k]
    return None


def _extract_popt(p_dict, key):
    """Extracts popt array whether stored directly or as (popt, pcov)."""
    if p_dict is None:
        return None
    val = p_dict.get(key, p_dict.get(key.replace("_reco", ""), None))
    if val is None:
        return None
    return val[0] if isinstance(val, (tuple, list)) and len(val) == 2 else val


def _safe_eval(func, x_arr):
    """Evaluate a ROOT TF1 (or any callable with .Eval) over an array, guarding against
    NaN/inf so a single bad point doesn't break the normalization / plot."""
    y = np.array([func.Eval(x) for x in x_arr], dtype=float)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    return y


def _langau_pdf(x_arr, width, mpv, area, gsigma, n_steps=100, sc=5.0):
    """Direct re-implementation of the classic ROOT 'langaufun' Landau (x) Gaussian
    convolution, evaluated natively in Python/NumPy instead of via a cloned ROOT TF1.
    """
    import ROOT  # noqa: local import

    invsq2pi = 0.3989422804014327
    mpshift = -0.22278298
    mpc = mpv - mpshift * width
    half_steps = max(int(n_steps // 2), 1)

    x_arr = np.asarray(x_arr, dtype=float)
    y = np.empty_like(x_arr)

    for idx, x0 in enumerate(x_arr):
        xlow = x0 - sc * gsigma
        xupp = x0 + sc * gsigma
        step = (xupp - xlow) / n_steps
        s = 0.0
        for i in range(1, half_steps + 1):
            xx1 = xlow + (i - 0.5) * step
            s += (ROOT.TMath.Landau(xx1, mpc, width) / width) * ROOT.TMath.Gaus(x0, xx1, gsigma)
            xx2 = xupp - (i - 0.5) * step
            s += (ROOT.TMath.Landau(xx2, mpc, width) / width) * ROOT.TMath.Gaus(x0, xx2, gsigma)
        y[idx] = area * step * s * invsq2pi / gsigma

    return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)


def inspect_all_rr_slices(
    data_hit_dfs,
    mc_hit_dfs,
    data_fit_params=None,
    mc_fit_params=None,
    rr_start=4,
    rr_end=40,
    rr_step=1,
    tpc=0,
    dedx_col="dedx",
    hist_nbins=150,
    hist_xmin=0.0,
    hist_xmax=10.0,
    first_stage_range=(0.5, 20.0),
    max_par_err=1.0,
    hist_alpha=0.35,
    mc_hist_alpha=0.18,
):
    """Loops over 1 cm RR slices, prints parameter comparisons, and plots normalized Data
    vs. MC slice fits alongside power-law evaluated global model curves over the full range.
    """
    n_planes = len(data_hit_dfs)
    rr_ranges = [(r, r + rr_step) for r in range(rr_start, rr_end, rr_step)]

    color_data = "#1f77b4"  # Blue
    color_mc = "#d62728"    # Red

    bin_edges = np.linspace(hist_xmin, hist_xmax, hist_nbins + 1)

    for rr_min, rr_max in rr_ranges:
        rr_center = 0.5 * (rr_min + rr_max)
        print("=" * 82)
        print(f"  RR SLICE: {rr_min:.1f} <= RR < {rr_max:.1f} cm | Center: {rr_center:.2f} cm | TPC: {tpc}")
        print("=" * 82)

        fig = plt.figure(figsize=(8.5 * n_planes, 7.5))
        gs = gridspec.GridSpec(1, n_planes, wspace=0.25)
        x_eval = np.linspace(hist_xmin, hist_xmax, 500)

        for plane_idx in range(n_planes):
            ax = fig.add_subplot(gs[0, plane_idx])
            df_data = data_hit_dfs[plane_idx] if plane_idx < len(data_hit_dfs) else None
            df_mc = mc_hit_dfs[plane_idx] if plane_idx < len(mc_hit_dfs) else None

            if df_data is None and df_mc is None:
                ax.set_title(f"Plane {plane_idx}: No Data/MC", fontsize=15, pad=10)
                continue

            handles_labels = []
            print(f"\n--- Plane {plane_idx} ---")

            # ==========================================
            # 1. PROCESS DATA
            # ==========================================
            func_d_ref = None
            if df_data is not None:
                sub_data = df_data if tpc == -1 else df_data[df_data["tpc"] == tpc]
                sub_data = sub_data[(sub_data["rr"] >= rr_min) & (sub_data["rr"] < rr_max)]

                if len(sub_data) >= 10:
                    hname_d = f"h_data_p{plane_idx}_rr{rr_center:.1f}"
                    hist_d = th1_from_series(
                        sub_data[dedx_col], hname_d, "", hist_nbins, hist_xmin, hist_xmax
                    )

                    counts_d = np.array([hist_d.GetBinContent(i) for i in range(1, hist_nbins + 1)])
                    max_d = np.max(counts_d) if np.max(counts_d) > 0 else 1.0
                    counts_d_norm = counts_d / max_d

                    d_fill = ax.stairs(
                        counts_d_norm,
                        bin_edges,
                        fill=True,
                        color=color_data,
                        alpha=hist_alpha,
                        label=rf"Data Hits ($N = {len(sub_data)}$)",
                    )
                    ax.stairs(
                        counts_d_norm,
                        bin_edges,
                        color=color_data,
                        linewidth=1.1,
                        alpha=0.9,
                    )
                    handles_labels.append(d_fill)

                    fit_res_d = langau_fit_two_stage(
                        hist_d,
                        first_stage_range=first_stage_range,
                        max_par_err=max_par_err,
                    )

                    if fit_res_d is not None and "func" in fit_res_d:
                        func_d_ref = fit_res_d["func"]
                        y_fit_d = _safe_eval(func_d_ref, x_eval)
                        max_yf_d = np.max(y_fit_d) if np.max(y_fit_d) > 0 else 1.0

                        w_d_fit = fit_res_d["pars"][0]
                        mpv_d_fit = fit_res_d["pars"][1]
                        g_d_fit = fit_res_d["pars"][3] if len(fit_res_d["pars"]) > 3 else np.nan
                        err_d = fit_res_d["errs"][1]

                        print(f"  [DATA Slice Fit Extracted]")
                        print(f"    MPV:    {mpv_d_fit:.4f} ± {err_d:.4f} | Width: {w_d_fit:.4f} | GSigma: {g_d_fit:.4f}")

                        (d_fit_line,) = ax.plot(
                            x_eval,
                            y_fit_d / max_yf_d,
                            "-",
                            color=color_data,
                            linewidth=2.2,
                            alpha=0.9,
                            label=f"Data Slice Fit (MPV = {mpv_d_fit:.2f} ± {err_d:.2f})",
                        )
                        handles_labels.append(d_fit_line)

            # --- Data Global Model Prediction ---
            p_dict_d = _get_params_dict(data_fit_params, plane_idx, tpc)
            popt_mpv_d = _extract_popt(p_dict_d, "mpv_reco")
            popt_w_d = _extract_popt(p_dict_d, "width")
            popt_g_d = _extract_popt(p_dict_d, "gsigma")

            if (
                popt_mpv_d is not None
                and popt_w_d is not None
                and popt_g_d is not None
                and func_d_ref is not None
            ):
                # Evaluated via Power Law formula: a * x^(-b) + c
                pred_mpv_d = power_law(rr_center, *popt_mpv_d)
                pred_w_d = power_law(rr_center, *popt_w_d)
                pred_g_d = power_law(rr_center, *popt_g_d)

                print(f"  [DATA Global Model Predicted @ RR = {rr_center:.2f} cm]")
                print(
                    f"    MPV:    {pred_mpv_d:.4f}        | Width: {pred_w_d:.4f} | GSigma:"
                    f" {pred_g_d:.4f}"
                )

                y_pred_d = _langau_pdf(x_eval, pred_w_d, pred_mpv_d, 1.0, pred_g_d)
                max_yp_d = np.max(y_pred_d) if np.max(y_pred_d) > 0 else 1.0

                (d_exp_line,) = ax.plot(
                    x_eval,
                    y_pred_d / max_yp_d,
                    ":",
                    color="#003366",
                    linewidth=2.2,
                    label=f"Data Model Pred (MPV = {pred_mpv_d:.2f})",
                )
                handles_labels.append(d_exp_line)

            # ==========================================
            # 2. PROCESS MC
            # ==========================================
            func_m_ref = None
            if df_mc is not None:
                sub_mc = df_mc if tpc == -1 else df_mc[df_mc["tpc"] == tpc]
                sub_mc = sub_mc[(sub_mc["rr"] >= rr_min) & (sub_mc["rr"] < rr_max)]

                if len(sub_mc) >= 10:
                    hname_m = f"h_mc_p{plane_idx}_rr{rr_center:.1f}"
                    hist_m = th1_from_series(
                        sub_mc[dedx_col], hname_m, "", hist_nbins, hist_xmin, hist_xmax
                    )

                    counts_m = np.array([hist_m.GetBinContent(i) for i in range(1, hist_nbins + 1)])
                    max_m = np.max(counts_m) if np.max(counts_m) > 0 else 1.0
                    counts_m_norm = counts_m / max_m

                    m_fill = ax.stairs(
                        counts_m_norm,
                        bin_edges,
                        fill=True,
                        color=color_mc,
                        alpha=mc_hist_alpha,
                        label=rf"MC Hits ($N = {len(sub_mc)}$)",
                    )
                    handles_labels.append(m_fill)

                    fit_res_m = langau_fit_two_stage(
                        hist_m,
                        first_stage_range=first_stage_range,
                        max_par_err=max_par_err,
                    )

                    if fit_res_m is not None and "func" in fit_res_m:
                        func_m_ref = fit_res_m["func"]
                        y_fit_m = _safe_eval(func_m_ref, x_eval)
                        max_yf_m = np.max(y_fit_m) if np.max(y_fit_m) > 0 else 1.0

                        w_m_fit = fit_res_m["pars"][0]
                        mpv_m_fit = fit_res_m["pars"][1]
                        g_m_fit = fit_res_m["pars"][3] if len(fit_res_m["pars"]) > 3 else np.nan
                        err_m = fit_res_m["errs"][1]

                        print(f"  [MC Slice Fit Extracted]")
                        print(f"    MPV:    {mpv_m_fit:.4f} ± {err_m:.4f} | Width: {w_m_fit:.4f} | GSigma: {g_m_fit:.4f}")

                        (m_fit_line,) = ax.plot(
                            x_eval,
                            y_fit_m / max_yf_m,
                            "--",
                            color=color_mc,
                            linewidth=2.2,
                            alpha=0.9,
                            label=f"MC Slice Fit (MPV = {mpv_m_fit:.2f} ± {err_m:.2f})",
                        )
                        handles_labels.append(m_fit_line)

            # --- MC Global Model Prediction ---
            p_dict_m = _get_params_dict(mc_fit_params, plane_idx, tpc)
            popt_mpv_m = _extract_popt(p_dict_m, "mpv_reco")
            popt_w_m = _extract_popt(p_dict_m, "width")
            popt_g_m = _extract_popt(p_dict_m, "gsigma")

            if (
                popt_mpv_m is not None
                and popt_w_m is not None
                and popt_g_m is not None
                and func_m_ref is not None
            ):
                # Evaluated via Power Law formula: a * x^(-b) + c
                pred_mpv_m = power_law(rr_center, *popt_mpv_m)
                pred_w_m = power_law(rr_center, *popt_w_m)
                pred_g_m = power_law(rr_center, *popt_g_m)

                print(f"  [MC Global Model Predicted @ RR = {rr_center:.2f} cm]")
                print(
                    f"    MPV:    {pred_mpv_m:.4f}        | Width: {pred_w_m:.4f} | GSigma:"
                    f" {pred_g_m:.4f}"
                )

                y_pred_m = _langau_pdf(x_eval, pred_w_m, pred_mpv_m, 1.0, pred_g_m)
                max_yp_m = np.max(y_pred_m) if np.max(y_pred_m) > 0 else 1.0

                (m_exp_line,) = ax.plot(
                    x_eval,
                    y_pred_m / max_yp_m,
                    ":",
                    color="#8b0000",
                    linewidth=2.2,
                    label=f"MC Model Pred (MPV = {pred_mpv_m:.2f})",
                )
                handles_labels.append(m_exp_line)

            # Subplot Styling
            ax.set_title(f"Plane {plane_idx}", fontsize=16, pad=10)
            ax.set_xlabel(r"Hit $dE/dx$ [MeV/cm]", fontsize=14)
            ax.set_xlim(hist_xmin, hist_xmax)
            ax.set_ylim(0, 1.25)
            ax.tick_params(axis="both", which="major", labelsize=11)
            ax.grid(True, linestyle=":", alpha=0.5)

            if plane_idx == 0:
                ax.set_ylabel("Normalized Amplitude (Peak = 1.0)", fontsize=14)

            if handles_labels:
                ax.legend(
                    handles=handles_labels,
                    loc="upper right",
                    fontsize=9,
                    framealpha=0.9,
                    labelspacing=0.5,
                    handlelength=2.0,
                )

            ax.set_box_aspect(1)

        tpc_str = f"TPC {tpc}" if tpc != -1 else "TPC Combined"
        fig.suptitle(
            rf"Data vs. MC LanGau Fits | {tpc_str} | Slice: {rr_min} $\leq$ RR < {rr_max} cm",
            fontsize=18,
            y=1.02,
        )
        plt.tight_layout()
        plt.show()

In [ ]:
mc_hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
data_hit_dfs = [data_hit0_df, data_hit1_df, data_hit2_df]

inspect_all_rr_slices(
    data_hit_dfs=data_hit_dfs,
    mc_hit_dfs=mc_hit_dfs,
    data_fit_params=data_params,
    mc_fit_params=mc_params,
    rr_start=2,
    rr_end=20,
    rr_step=1,
    tpc=-1,  # Both TPCs combined
)

In [ ]:
# ===========================================================================
# Generate C++ literals for the mpv / gsigma / width maps (x MC and Data),
# from fit_params dicts returned by analyze_theoretical():
#   fit_params: dict[(plane, tpc)] -> {'mpv_reco': (popt, perr),
#                                       'gsigma':   (popt, perr),
#                                       'width':    (popt, perr)}
#
# Only overwrites the entries for the pdg you just fit, and even then only
# where a fit actually converged this run -- anything that fit=None (failed
# convergence), wasn't the target pdg, or wasn't refit at all this run keeps
# whatever's in EXISTING_ALL_TXT below.
#
# To update before your next run: copy ALL SIX current PhysdEdx.cpp map
# blocks straight out of the .cpp file as one paste into EXISTING_ALL_TXT
# below -- no need to split them apart, no manual re-typing into dicts.
# ===========================================================================

import re

N_PLANES = 3
PDG_LIST = [13, 2212, 211]   # keep this order to match existing map style
TPC = -1                     # -1 = combined TPCs; the arg you're using downstream

# Map from parameter name (as it appears in fit_params) -> the PhysdEdx.cpp
# variable name for MC and for Data. EDIT THESE to match whatever names
# actually appear in your PhysdEdx.cpp -- parse_all_cpp_maps() below picks up
# any `PhysdEdx::<name> = { ... };` block automatically regardless of name,
# it just needs to match what you list here so the right existing map gets
# used as the fallback for each parameter.
CPP_VAR_NAMES = {
    "mpv_reco": {"mc": "pdg_plane_mpv_map", "data": "pdg_plane_mpv_map_data"},
    "gsigma":   {"mc": "pdg_plane_gsigma_map", "data": "pdg_plane_gsigma_map_data"},
    "width":    {"mc": "pdg_plane_width_map", "data": "pdg_plane_width_map_data"},
}


def find_balanced_brace(text, open_idx):
    """Given text[open_idx] == '{', return the index of its matching '}',
    correctly handling arbitrary nesting depth (regex alone can't do this)."""
    assert text[open_idx] == '{', f"expected '{{' at index {open_idx}, got {text[open_idx]!r}"
    depth = 0
    for i in range(open_idx, len(text)):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return i
    raise ValueError("Unbalanced braces in C++ text -- check the pasted block")


def parse_cpp_map(cpp_text):
    """Parse a single 'vector<std::map<int, vector<double>>> ... = { ... };' body
    (the {...} block itself, no outer 'vector<...> Name =' wrapper) and
    return dict[(plane, pdg)] -> (a, b, c). Uses explicit brace-depth
    tracking instead of regex, since the structure is 3 levels deep
    (vector{ plane{ entry{ triplet{} } } }) and Python's re can't match
    nested braces reliably."""
    text = cpp_text.strip()
    assert text.startswith('{'), "expected block to start with '{'"
    outer_end = find_balanced_brace(text, 0)
    outer_content = text[1:outer_end]

    result = {}
    plane_idx = -1
    i = 0
    n = len(outer_content)
    while i < n:
        if outer_content[i] == '{':
            plane_end = find_balanced_brace(outer_content, i)
            plane_text = outer_content[i + 1:plane_end]
            plane_idx += 1

            j = 0
            m = len(plane_text)
            while j < m:
                if plane_text[j] == '{':
                    entry_end = find_balanced_brace(plane_text, j)
                    entry_text = plane_text[j + 1:entry_end]  # e.g. "211, {0.18325785, 0.0030161048, 3.2302248}"
                    entry_match = re.match(
                        r'\s*(-?\d+)\s*,\s*\{([^{}]*)\}\s*$', entry_text
                    )
                    if entry_match:
                        pdg_str, triplet_str = entry_match.groups()
                        parts = [p.strip() for p in triplet_str.split(',')]
                        if len(parts) == 3:
                            try:
                                a, b, c = (float(p) for p in parts)
                                result[(plane_idx, int(pdg_str))] = (a, b, c)
                            except ValueError:
                                pass
                    j = entry_end + 1
                else:
                    j += 1
            i = plane_end + 1
        else:
            i += 1
    return result


def parse_all_cpp_maps(all_text):
    """Splits one big pasted chunk containing multiple
    'vector<std::map<int, vector<double>>> PhysdEdx::<name> = { ... };'
    blocks into dict[name] -> parse_cpp_map(...) result, keyed by the
    variable name (e.g. 'pdg_plane_mpv_map', 'pdg_plane_mpv_map_data')."""
    maps = {}
    for m in re.finditer(r'PhysdEdx::(\w+)\s*=\s*', all_text):
        name = m.group(1)
        brace_start = all_text.index('{', m.end())
        brace_end = find_balanced_brace(all_text, brace_start)
        body = all_text[brace_start:brace_end + 1]
        maps[name] = parse_cpp_map(body)
    return maps


# --- Paste ALL SIX current PhysdEdx.cpp blocks here as one chunk, in any order.
# If you don't have dedicated mpv/gsigma/width maps yet (e.g. you're introducing
# them for the first time), just leave this empty ("") -- every entry will fall
# back to -1, -1, -1 until you've run a fit for it at least once.
EXISTING_ALL_TXT = """
vector<std::map<int, vector<double>>> PhysdEdx::pdg_plane_mpv_map = {
  // Plane 0
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {2.6701971, 0.10299393, 1.9153627}}
  },
  // Plane 1
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {2.6389293, 0.10258216, 1.9046574}}
  },
  // Plane 2
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {2.6437039, 0.10025529, 1.9157184}}
  }
};

vector<std::map<int, vector<double>>> PhysdEdx::pdg_plane_gsigma_map = {
  // Plane 0
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.65608695, 0.17877152, 0.24958494}}
  },
  // Plane 1
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.59223739, 0.15236135, 0.29807549}}
  },
  // Plane 2
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.7702815, 0.19677627, 0.15255084}}
  }
};

vector<std::map<int, vector<double>>> PhysdEdx::pdg_plane_width_map = {
  // Plane 0
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.035518428, 0.097570026, 0.098712514}}
  },
  // Plane 1
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.070367348, 0.17737357, 0.11468284}}
  },
  // Plane 2
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.060291702, 0.053821677, 0.092030872}}
  }
};

vector<std::map<int, vector<double>>> PhysdEdx::pdg_plane_mpv_map_data = {
  // Plane 0
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {2.8221594, 0.10893512, 1.9640755}}
  },
  // Plane 1
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {2.7024618, 0.10716608, 1.9254381}}
  },
  // Plane 2
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {2.6647518, 0.10396786, 1.8798164}}
  }
};

vector<std::map<int, vector<double>>> PhysdEdx::pdg_plane_gsigma_map_data = {
  // Plane 0
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.57301461, 0.14917845, 0.24447239}}
  },
  // Plane 1
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.66779108, 0.1705544, 0.30402557}}
  },
  // Plane 2
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.91424658, 0.20613023, 0.12400119}}
  }
};

vector<std::map<int, vector<double>>> PhysdEdx::pdg_plane_width_map_data = {
  // Plane 0
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.24899924, 0.31087357, 0.12549702}}
  },
  // Plane 1
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.072910702, 0.091682912, 0.14279511}}
  },
  // Plane 2
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {0.073582868, 0.0492103, 0.098276045}}
  }
};
"""

_all_maps = parse_all_cpp_maps(EXISTING_ALL_TXT)
print("Parsed entry counts:",
      {k: len(v) for k, v in _all_maps.items()})  # sanity check -- should be 9 each (3 planes x 3 pdgs)


def fmt_triplet(popt):
    if popt is None:
        return "-1, -1, -1"
    a, b, c = popt
    return f"{a:.8g}, {b:.8g}, {c:.8g}"


def build_map_literal(var_name, existing_map, fit_params, target_pdg, tpc=-1):
    """existing_map: dict[(plane, pdg)] -> (a,b,c) triplet, used as the source
    of truth for any (plane, pdg) not being freshly updated -- including the
    target pdg itself when this run's fit for that plane didn't converge.
    fit_params: dict[(plane, tpc)] -> (popt, perr), as returned by analyze_theoretical."""
    lines = [f"vector<std::map<int, vector<double>>> PhysdEdx::{var_name} = {{"]
    for plane in range(N_PLANES):
        lines.append(f"  // Plane {plane}")
        lines.append("  {")
        for i_pdg, this_pdg in enumerate(PDG_LIST):
            if this_pdg == target_pdg:
                popt, perr = fit_params.get((plane, tpc), (None, None))
                if popt is not None:
                    triplet = fmt_triplet(popt)
                else:
                    triplet = fmt_triplet(existing_map.get((plane, this_pdg)))
                    print(f"  [warn] no fit for plane={plane}, tpc={tpc}, pdg={this_pdg} "
                          f"-> keeping existing value ({triplet})")
            else:
                triplet = fmt_triplet(existing_map.get((plane, this_pdg)))
            comma = "," if i_pdg < len(PDG_LIST) - 1 else ""
            lines.append(f"    {{{this_pdg}, {{{triplet}}}}}{comma}")
        comma_plane = "," if plane < N_PLANES - 1 else ""
        lines.append(f"  }}{comma_plane}")
    lines.append("};")
    return "\n".join(lines)


def split_fit_params_by_key(fit_params):
    """fit_params: dict[(plane, tpc)] -> {param_name: (popt, perr), ...}
    Returns: dict[param_name] -> dict[(plane, tpc)] -> (popt, perr)

    e.g. split_fit_params_by_key(mc_fit_params)['mpv_reco'] has the same
    shape build_map_literal() already expects for its `fit_params` arg.
    """
    if fit_params is None:
        return {}

    out = {}
    for key, param_dict in fit_params.items():
        if param_dict is None:
            continue
        for pname, val in param_dict.items():
            out.setdefault(pname, {})[key] = val
    return out


def update_all_param_maps(mc_fit_params, data_fit_params, target_pdg, all_maps=None, tpc=TPC):
    """Ties it all together: splits mc_fit_params/data_fit_params by parameter
    name, looks up the matching existing map (parsed from EXISTING_ALL_TXT via
    CPP_VAR_NAMES) as the fallback source of truth, and returns
    dict[cpp_var_name] -> literal string for all 6 mpv/gsigma/width x mc/data
    blocks. Prints nothing itself -- caller decides whether/how to print."""
    if all_maps is None:
        all_maps = _all_maps

    mc_split = split_fit_params_by_key(mc_fit_params)
    data_split = split_fit_params_by_key(data_fit_params)

    literals = {}
    for pname, names in CPP_VAR_NAMES.items():
        mc_var = names["mc"]
        data_var = names["data"]

        literals[mc_var] = build_map_literal(
            mc_var,
            all_maps.get(mc_var, {}),
            mc_split.get(pname, {}),
            target_pdg,
            tpc=tpc,
        )
        literals[data_var] = build_map_literal(
            data_var,
            all_maps.get(data_var, {}),
            data_split.get(pname, {}),
            target_pdg,
            tpc=tpc,
        )
    return literals


# --- Run it: builds and prints all 6 updated C++ blocks, ready to paste back
# into PhysdEdx.cpp. Existing (plane, pdg) entries not covered by this run's
# fit (wrong pdg, or a convergence failure for the target pdg) are carried
# over unchanged from EXISTING_ALL_TXT.
_literals = update_all_param_maps(mc_params, data_params, pdg, tpc=TPC)
for _var_name in (
    "pdg_plane_mpv_map", "pdg_plane_gsigma_map", "pdg_plane_width_map",
    "pdg_plane_mpv_map_data", "pdg_plane_gsigma_map_data", "pdg_plane_width_map_data",
):
    print(_literals[_var_name])
    print()

In [ ]:
import re
import numpy as np
import pandas as pd

# ===========================================================================
# Configuration
# ===========================================================================
N_PLANES = 3
PDG_LIST = [13, 2212, 211]
TPC = -1

# Variable mapping: (rr_center, dataset_type) -> C++ variable name
MAP_NAMES = {
    (2.5, "mc"):   "pdg_plane_low_rr_2p5",
    (2.5, "data"): "pdg_plane_low_rr_2p5_data",
    (3.5, "mc"):   "pdg_plane_low_rr_3p5",
    (3.5, "data"): "pdg_plane_low_rr_3p5_data",
}

# ===========================================================================
# Paste your existing C++ map blocks here to update incrementally.
# Leave as empty string "" if running for the first time.
# ===========================================================================
EXISTING_ALL_TXT = """
vector<std::map<int, vector<double>>> PhysdEdx::pdg_plane_low_rr_3p5 = {
  // Plane 0
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {-1, -1, -1}}
  },
  // Plane 1
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {-1, -1, -1}}
  },
  // Plane 2
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {-1, -1, -1}}
  }
};

vector<std::map<int, vector<double>>> PhysdEdx::pdg_plane_low_rr_3p5_data = {
  // Plane 0
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {4.126522, 0.6841, 0.203041}}
  },
  // Plane 1
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {-1, -1, -1}}
  },
  // Plane 2
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {-1, -1, -1}}
  }
};

vector<std::map<int, vector<double>>> PhysdEdx::pdg_plane_low_rr_4p5 = {
  // Plane 0
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {-1, -1, -1}}
  },
  // Plane 1
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {-1, -1, -1}}
  },
  // Plane 2
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {-1, -1, -1}}
  }
};

vector<std::map<int, vector<double>>> PhysdEdx::pdg_plane_low_rr_4p5_data = {
  // Plane 0
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {3.75795, 0.556428, 0.190533}}
  },
  // Plane 1
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {-1, -1, -1}}
  },
  // Plane 2
  {
    {13, {-1, -1, -1}},
    {2212, {-1, -1, -1}},
    {211, {-1, -1, -1}}
  }
};
"""

# ===========================================================================
# C++ Parser Utilities
# ===========================================================================
def find_balanced_brace(text, open_idx):
    assert text[open_idx] == '{'
    depth = 0
    for i in range(open_idx, len(text)):
        if text[i] == '{': depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0: return i
    raise ValueError("Unbalanced braces in pasted C++ text.")

def parse_cpp_map(cpp_text):
    """Parses C++ map into dict[(plane, pdg)] -> 3-element list of floats."""
    text = cpp_text.strip()
    if not text.startswith('{'): return {}
    outer_end = find_balanced_brace(text, 0)
    outer_content = text[1:outer_end]

    result = {}
    plane_idx = -1
    i = 0
    n = len(outer_content)
    while i < n:
        if outer_content[i] == '{':
            plane_end = find_balanced_brace(outer_content, i)
            plane_text = outer_content[i + 1:plane_end]
            plane_idx += 1

            j = 0
            m = len(plane_text)
            while j < m:
                if plane_text[j] == '{':
                    entry_end = find_balanced_brace(plane_text, j)
                    entry_text = plane_text[j + 1:entry_end]
                    entry_match = re.match(r'\s*(-?\d+)\s*,\s*\{([^{}]*)\}\s*$', entry_text)
                    if entry_match:
                        pdg_str, vec_str = entry_match.groups()
                        vals = [float(p.strip()) for p in vec_str.split(',')]
                        result[(plane_idx, int(pdg_str))] = vals
                    j = entry_end + 1
                else: j += 1
            i = plane_end + 1
        else: i += 1
    return result

def parse_all_cpp_maps(all_text):
    maps = {}
    for m in re.finditer(r'PhysdEdx::(\w+)\s*=\s*', all_text):
        name = m.group(1)
        brace_start = all_text.index('{', m.end())
        brace_end = find_balanced_brace(all_text, brace_start)
        maps[name] = parse_cpp_map(all_text[brace_start:brace_end + 1])
    return maps

# ===========================================================================
# Data Extraction & Literal Builder
# ===========================================================================
def extract_single_rr_vector(slice_df_dict, target_rr):
    """
    Extracts [mpv, gsigma, width] for a specific target_rr.
    Returns dict[(plane, tpc)] -> [mpv, gsigma, width]
    """
    extracted = {}
    if not slice_df_dict:
        return extracted

    for key, df in slice_df_dict.items():
        if df is None or (isinstance(df, pd.DataFrame) and df.empty):
            continue
        row = df[np.isclose(df["rr_center"], target_rr)]
        if len(row) > 0:
            r = row.iloc[0]
            extracted[key] = [r["mpv_reco"], r["gsigma"], r["width"]]
        else:
            extracted[key] = [-1.0, -1.0, -1.0]
    return extracted

def fmt_vector(vec):
    if not vec or len(vec) == 0:
        return "-1, -1, -1"
    return ", ".join(f"{x:.8g}" for x in vec)

def build_3val_map_literal(var_name, existing_map, new_fit_dict, target_pdg, tpc=TPC):
    """
    Builds C++ literal vector block containing {mpv, gsigma, width}.
    """
    lines = [f"vector<std::map<int, vector<double>>> PhysdEdx::{var_name} = {{"]
    for plane in range(N_PLANES):
        lines.append(f"  // Plane {plane}")
        lines.append("  {")
        for i_pdg, this_pdg in enumerate(PDG_LIST):
            if this_pdg == target_pdg:
                new_vec = new_fit_dict.get((plane, tpc), None)
                prev_vec = existing_map.get((plane, this_pdg), [-1.0, -1.0, -1.0])
                if len(prev_vec) < 3: prev_vec = [-1.0, -1.0, -1.0]

                if new_vec and len(new_vec) == 3:
                    merged = [
                        new_vec[i] if new_vec[i] != -1.0 else prev_vec[i]
                        for i in range(3)
                    ]
                else:
                    merged = prev_vec
                vec_str = fmt_vector(merged)
            else:
                existing_vec = existing_map.get((plane, this_pdg), [-1.0, -1.0, -1.0])
                vec_str = fmt_vector(existing_vec)

            comma = "," if i_pdg < len(PDG_LIST) - 1 else ""
            lines.append(f"    {{{this_pdg}, {{{vec_str}}}}}{comma}")
        comma_plane = "," if plane < N_PLANES - 1 else ""
        lines.append(f"  }}{comma_plane}")
    lines.append("};")
    return "\n".join(lines)

def update_3val_low_rr_maps(mc_slice_dfs, data_slice_dfs, target_pdg, tpc=TPC, existing_txt=EXISTING_ALL_TXT):
    """
    Main entry point: updates all four maps (MC/Data for 3.5cm and 4.5cm).
    """
    parsed_existing = parse_all_cpp_maps(existing_txt)
    literals = {}

    for (target_rr, sample_type), var_name in MAP_NAMES.items():
        slice_dfs = mc_slice_dfs if sample_type == "mc" else data_slice_dfs
        extracted_fits = extract_single_rr_vector(slice_dfs, target_rr)
        existing_map = parsed_existing.get(var_name, {})

        literals[var_name] = build_3val_map_literal(
            var_name, existing_map, extracted_fits, target_pdg, tpc=tpc
        )

    return literals

# ===========================================================================
# Execution Example
# ===========================================================================
if __name__ == "__main__":
    _literals = update_3val_low_rr_maps(
        mc_slice_dfs=mc_results,           # Replace with mc_results_dict if available
        data_slice_dfs=data_results, # Pass your data results dict here
        target_pdg=211,
        tpc=TPC
    )

    for _name in MAP_NAMES.values():
        print(_literals[_name])
        print("\n")